# Chinese world — Cultura vs Cross-Verified, what is the difference?

Take **all individuals who could plausibly belong to the Chinese world** in each database, and look at who is in one but not the other.

- **Cultura side** — every individual whose `polity_name` (semicolon list) contains any canonical Chinese dynasty (Shang → Qing).
- **Cross-Verified side** — every individual whose `citizenship_1_b` *or* `citizenship_2_b` falls into the **Chinese world citizenship set**, derived from Cultura's own `polities_modern_countries_cliopatria` table (the polities map to: People's Republic of China, Taiwan, North Korea, Mongolia, Russia — those that have CV equivalents are kept).

What we then show:
1. Total counts in each database, the Q-id intersection, and the two set differences (Cultura-only, CV-only).
2. Per-century counts and the difference per century.
3. A few example individuals on each side of the difference, so it's clear what kind of people each database is adding."

## 1. Configuration

In [ ]:
import duckdb
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

DB_PATH = '../data/humans_clean.duckdb'
CV_PATH = '../data/similar_databases/cross-verified-database/cross-verified-database.utf8.csv.gz'

CENTURY_MIN, CENTURY_MAX = -16, 20  # −1600 BCE → 2000 CE

COL_CULTURA = '#2f5b8a'
COL_CV      = '#b5542a'

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'font.size': 12,
    'font.family': 'DejaVu Sans',
})

## 2. Chinese world definitions

Cultura: 25 canonical dynasties. CV: derived from Cultura's `polities_modern_countries_cliopatria` mapping plus CV-side aliases (`People's Republic of China` → `China`, etc.) — keeping only entries that exist in CV's citizenship vocabulary.

In [ ]:
CHINESE_DYNASTIES = [
    'Shang Dynasty', 'Zhou Dynasty', 'Qin Dynasty', 'Han Dynasty', 'Xin Dynasty',
    'Western Jin', 'Eastern Jin', 'Liu Song Dynasty', 'Liang Dynasty', 'Chen Dynasty',
    'Northern Wei', 'Eastern Wei', 'Western Wei', 'Northern Zhou', 'Northern Qi',
    'Sui Dynasty', 'Tang Dynasty', 'Five Dynasties and Ten Kingdoms',
    'Northern Song', 'Southern Song', 'Liao Dynasty', 'Western Xia',
    'Yuan Dynasty', 'Ming Dynasty', 'Qing Dynasty',
]

# Aliases — Cultura country_name → CV citizenship_1_b
CULTURA_TO_CV_ALIAS = {
    "People's Republic of China": 'China',
    'North Korea':                 'North_Korea',
}

def cultura_country_to_cv(name):
    if name is None:
        return None
    return CULTURA_TO_CV_ALIAS.get(name, name.replace(' ', '_'))

# Look up the modern-country mapping for the dynasties
con = duckdb.connect(DB_PATH, read_only=True)
ph = ','.join('?' * len(CHINESE_DYNASTIES))
country_table = con.execute(f"""
    SELECT DISTINCT polity_name, country_name
    FROM polities_modern_countries_cliopatria
    WHERE polity_name IN ({ph}) AND country_name IS NOT NULL
    """, CHINESE_DYNASTIES).pl()
country_table = country_table.with_columns(
    pl.col('country_name').map_elements(cultura_country_to_cv, return_dtype=pl.Utf8).alias('cv_citizenship')
)

# CV vocabulary — needed to drop translations CV does not have
cv_vocab = pl.read_csv(
    CV_PATH,
    columns=['citizenship_1_b', 'citizenship_2_b'],
)
CV_KNOWN = (
    set(cv_vocab.filter(pl.col('citizenship_1_b').is_not_null())['citizenship_1_b'].to_list())
    | set(cv_vocab.filter(pl.col('citizenship_2_b').is_not_null())['citizenship_2_b'].to_list())
)
del cv_vocab

country_table = country_table.with_columns(
    pl.col('cv_citizenship').is_in(CV_KNOWN).alias('in_cv_vocab')
)
CV_CHINESE_CITIZENSHIPS = set(
    country_table.filter(pl.col('in_cv_vocab'))['cv_citizenship'].to_list()
)
con.close()

print(f'Dynasties: {len(CHINESE_DYNASTIES)}')
print(f'Derived CV citizenships ({len(CV_CHINESE_CITIZENSHIPS)}): '
      f'{sorted(CV_CHINESE_CITIZENSHIPS)}')
country_table

## 3. Cultura — load Chinese individuals and assign a floruit year

Joins `individuals_cliopatria` (for the polity filter) with `individuals_floruit_period` (for `floruit_year`). Each individual is counted once.

In [ ]:
conn = duckdb.connect(DB_PATH, read_only=True)

like_clauses = ' OR '.join(["';' || ic.polity_name || ';' LIKE ?"] * len(CHINESE_DYNASTIES))
params = [f'%;{d};%' for d in CHINESE_DYNASTIES]

cultura = conn.execute(f"""
    SELECT DISTINCT ic.wikidata_id,
           fp.floruit_year,
           fp.floruit_period_start AS s,
           fp.floruit_period_end   AS e
    FROM individuals_cliopatria ic
    JOIN individuals_floruit_period fp USING (wikidata_id)
    WHERE ({like_clauses})
""", params).pl().unique(subset=['wikidata_id'])
conn.close()

# Use floruit_year if present, else midpoint of [start, end]
cultura = cultura.with_columns(
    pl.coalesce(
        pl.col('floruit_year'),
        ((pl.col('s') + pl.col('e')) / 2).round(0).cast(pl.Int64),
    ).alias('fy')
).filter(pl.col('fy').is_not_null()).with_columns(
    pl.col('fy').cast(pl.Int64),
    (pl.col('fy') // 100).alias('century').cast(pl.Int64),
)

n_cultura = cultura.height
print(f'Cultura Chinese-world individuals : {n_cultura:,}')

## 4. Cross-Verified — load Chinese individuals and assign a floruit year

Filter on `citizenship_1_b` or `citizenship_2_b`. Floruit year:
- if `birth` and `death` are both known → midpoint;
- if only `birth` → `birth + 30`;
- if only `death` → `death − 30`.

The same `+30` adult-onset convention is what the original Pantheon paper uses.

In [ ]:
cv = pl.read_csv(
    CV_PATH,
    columns=['wikidata_code', 'name', 'citizenship_1_b', 'citizenship_2_b',
             'birth', 'death', 'updated_death_date',
             'level1_main_occ', 'level2_main_occ', 'level3_main_occ',
             'list_wikipedia_editions'],
)

cv = cv.filter(
    pl.col('citizenship_1_b').is_in(CV_CHINESE_CITIZENSHIPS)
    | pl.col('citizenship_2_b').is_in(CV_CHINESE_CITIZENSHIPS)
).with_columns(
    pl.coalesce([pl.col('death'), pl.col('updated_death_date')]).alias('death_eff'),
).with_columns(
    pl.when(pl.col('birth').is_not_null() & pl.col('death_eff').is_not_null())
      .then((pl.col('birth') + pl.col('death_eff')) / 2)
    .when(pl.col('birth').is_not_null())
      .then(pl.col('birth') + 30)
    .when(pl.col('death_eff').is_not_null())
      .then(pl.col('death_eff') - 30)
    .otherwise(None)
    .alias('fy')
).filter(pl.col('fy').is_not_null()).unique(subset=['wikidata_code']).with_columns(
    pl.col('fy').round(0).cast(pl.Int64),
).with_columns(
    (pl.col('fy') // 100).alias('century').cast(pl.Int64),
)

n_cv = cv.height
print(f'Cross-Verified Chinese-world individuals : {n_cv:,}')

## 5. Total counts (overall and overlap)

In [ ]:
cultura_qids = set(cultura['wikidata_id'].to_list())
cv_qids      = set(cv.filter(pl.col('wikidata_code').is_not_null())['wikidata_code'].cast(pl.Utf8).to_list())

in_both = cultura_qids & cv_qids
cultura_only = cultura_qids - cv_qids
cv_only = cv_qids - cultura_qids

summary = pl.DataFrame({
    'metric': [
        'Cultura (Chinese polities)',
        'Cross-Verified (Chinese citizenships)',
        'Shared (Q-id overlap)',
        'Cultura only',
        'Cross-Verified only',
    ],
    'count': [n_cultura, n_cv, len(in_both), len(cultura_only), len(cv_only)],
})
summary

## 6. Where do the two databases differ?

For each individual on each side, we have a name, century, and main occupation (CV side) / `name_en` from `individuals_floruit_period` (Cultura side). Look at the *kind* of people present in one but not the other."

In [ ]:
# Enrich Cultura individuals with names
con = duckdb.connect(DB_PATH, read_only=True)
cultura_meta = con.execute("SELECT wikidata_id, name_en FROM individuals WHERE wikidata_id IN ({})".format(
        ','.join(['?'] * len(cultura_qids))
    ), list(cultura_qids)).pl()
con.close()
cultura_full = cultura.join(cultura_meta, on='wikidata_id', how='left')

# CV-only and Cultura-only views
cv_only_df      = cv.filter(pl.col('wikidata_code').is_in(cv_only))
cultura_only_df = cultura_full.filter(pl.col('wikidata_id').is_in(cultura_only))

print(f'Cultura-only        : {cultura_only_df.height:,}')
print(f'Cross-Verified-only : {cv_only_df.height:,}')

### 6.1 Difference per century — bar chart of unique additions

For each century, how many individuals does Cultura add that CV does not have, and vice-versa.

In [ ]:
centuries_arr = np.arange(CENTURY_MIN, CENTURY_MAX + 1)

def per_cent_counts(df_pl):
    counts = df_pl.group_by('century').agg(pl.len().alias('n'))
    by_c = dict(counts.iter_rows())
    return np.array([by_c.get(int(c), 0) for c in centuries_arr])

cu_only_per_c = per_cent_counts(cultura_only_df)
cv_only_per_c = per_cent_counts(cv_only_df)

fig, ax = plt.subplots(figsize=(11, 4.6))
w = 0.42
ax.bar(centuries_arr - w/2, cu_only_per_c, width=w,
       color=COL_CULTURA, label=f'Cultura only  ({cultura_only_df.height:,})')
ax.bar(centuries_arr + w/2, cv_only_per_c, width=w,
       color=COL_CV,      label=f'Cross-Verified only  ({cv_only_df.height:,})')
ax.set_xlabel('Century (year // 100, BCE ← → CE)')
ax.set_ylabel('Individuals exclusive to one DB')
ax.set_title('Chinese world — what each database adds, per century', loc='left', pad=12)
ax.set_xlim(CENTURY_MIN, CENTURY_MAX)
ax.legend(loc='upper left', frameon=False)
fig.tight_layout()
plt.show()

### 6.2 What kind of people does Cross-Verified contribute that Cultura misses?

Distribution of `level1_main_occ` for the CV-only set.

In [ ]:
occ_dist = (
    cv_only_df.with_columns(pl.col('level1_main_occ').fill_null('(unknown)'))
    .group_by('level1_main_occ').agg(pl.len().alias('CV-only individuals'))
    .sort('CV-only individuals', descending=True)
    .head(15)
)
occ_dist

### 6.3 Examples — 10 random individuals on each side of the difference

Use `SEED = 42` for reproducibility.

In [ ]:
SEED = 42
N_EX = 10

ex_cu = (
    cultura_only_df.sample(n=min(N_EX, cultura_only_df.height), seed=SEED)
    .select([
        pl.col('wikidata_id').alias('qid'),
        pl.col('name_en').alias('name'),
        pl.col('fy').alias('floruit_year'),
        pl.col('century'),
    ])
    .sort('floruit_year')
)

ex_cv = (
    cv_only_df.sample(n=min(N_EX, cv_only_df.height), seed=SEED)
    .select([
        pl.col('wikidata_code').alias('qid'),
        pl.col('name'),
        pl.col('fy').alias('floruit_year'),
        pl.col('century'),
        pl.col('level1_main_occ'),
        pl.col('level3_main_occ'),
        pl.col('citizenship_1_b'),
        pl.col('citizenship_2_b'),
    ])
    .sort('floruit_year')
)

print('=== Cultura-only — 10 examples ===')
print(ex_cu)
print('\n=== Cross-Verified-only — 10 examples ===')
print(ex_cv)

## 7. Per-century counts

In [ ]:
centuries = np.arange(CENTURY_MIN, CENTURY_MAX + 1)

def per_century(df_pl):
    counts = df_pl.group_by('century').agg(pl.len().alias('n'))
    by_c = dict(counts.iter_rows())
    return np.array([by_c.get(int(c), 0) for c in centuries])

by_c_cultura = per_century(cultura)
by_c_cv      = per_century(cv)

trends = pl.DataFrame({
    'century':  centuries.tolist(),
    'cultura':  by_c_cultura.tolist(),
    'cv':       by_c_cv.tolist(),
}).with_columns(
    (pl.col('cultura') / pl.col('cultura').sum()).alias('cultura_share'),
    (pl.col('cv') / pl.col('cv').sum()).alias('cv_share'),
)
trends.tail(15)

## 8. Figure — raw counts per century

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.6))
ax.plot(trends['century'].to_list(), trends['cultura'].to_list(), color=COL_CULTURA, lw=1.6,
        marker='o', ms=3.5, label=f'Cultura  (N = {n_cultura:,})')
ax.plot(trends['century'].to_list(), trends['cv'].to_list(), color=COL_CV, lw=1.6,
        marker='s', ms=3.5, label=f'Cross-Verified  (N = {n_cv:,})')
ax.set_xlabel('Century (year // 100, BCE ← → CE)')
ax.set_ylabel('Individuals')
ax.set_title('Chinese world — individuals per century', loc='left', pad=12)
ax.legend(loc='upper left', frameon=False)
ax.set_xlim(CENTURY_MIN, CENTURY_MAX)
fig.tight_layout()
plt.show()

## 9. Figure — normalized trends (share of each database's total)

Each series sums to 1.0, so the *shape* of the trajectories can be compared independently of the very different totals.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.6))
ax.plot(trends['century'].to_list(), (trends['cultura_share'] * 100).to_list(), color=COL_CULTURA,
        lw=1.6, marker='o', ms=3.5, label='Cultura')
ax.plot(trends['century'].to_list(), (trends['cv_share'] * 100).to_list(), color=COL_CV,
        lw=1.6, marker='s', ms=3.5, label='Cross-Verified')
ax.set_xlabel('Century (year // 100, BCE ← → CE)')
ax.set_ylabel('Share of database total (%)')
ax.set_title('Chinese world — normalized trends per century', loc='left', pad=12)
ax.legend(loc='upper left', frameon=False)
ax.set_xlim(CENTURY_MIN, CENTURY_MAX)
fig.tight_layout()
plt.show()

## 10. Figure — ratio Cultura / Cross-Verified per century

Where Cultura adds the most coverage relative to Cross-Verified.

In [ ]:
cultura_arr = trends['cultura'].to_numpy().astype(float)
cv_arr      = trends['cv'].to_numpy().astype(float)
ratio = np.divide(cultura_arr, cv_arr, out=np.full_like(cultura_arr, np.nan), where=cv_arr > 0)

fig, ax = plt.subplots(figsize=(11, 4.0))
ax.bar(trends['century'].to_list(), np.nan_to_num(ratio, nan=0.0), color='#7f7f7f', width=0.8)
ax.axhline(1.0, color='black', lw=0.7, ls='--')
ax.set_xlabel('Century (year // 100, BCE ← → CE)')
ax.set_ylabel('Cultura / Cross-Verified')
ax.set_title('Coverage ratio per century', loc='left', pad=12)
ax.set_xlim(CENTURY_MIN, CENTURY_MAX)
fig.tight_layout()
plt.show()

## 11. Export

In [ ]:
trends_path = '../data/chinese_world_cultura_vs_cv_century.csv'
trends.write_csv(trends_path)
print(f'wrote {trends_path}  ({trends.height:,} rows)')

cu_only_out = cultura_only_df.select([
    pl.col('wikidata_id').alias('qid'),
    pl.col('name_en').alias('name'),
    pl.col('fy').alias('floruit_year'),
    pl.col('century'),
])
cu_only_out.write_csv('../data/chinese_world_cultura_only.csv')
print(f'wrote ../data/chinese_world_cultura_only.csv  ({cu_only_out.height:,} rows)')

cv_only_out = cv_only_df.select([
    pl.col('wikidata_code').alias('qid'),
    pl.col('name'),
    pl.col('fy').alias('floruit_year'),
    pl.col('century'),
    pl.col('level1_main_occ'),
    pl.col('level3_main_occ'),
    pl.col('citizenship_1_b'),
    pl.col('citizenship_2_b'),
])
cv_only_out.write_csv('../data/chinese_world_cv_only.csv')
print(f'wrote ../data/chinese_world_cv_only.csv      ({cv_only_out.height:,} rows)')